# Clean run — current `mbo_utilities` venv

Run the suite2p pipeline with **all known confounds disabled**, then compare against the v2-7-7 sibling notebook.

Settings:
- `fix_phase = False` — disable bidirectional scan-phase correction (forced via the array attribute, bypassing any GUI/CLI plumbing)
- `use_fft = False` — disable FFT-based subpixel refinement on whatever phase work might still happen
- registration **off** (`do_registration = False`)
- `anatomical_only = 4` (cellpose on `max_proj`)
- `tau = 1.3` (LBM default — controls bin_size = round(tau*fs) = 18)
- `spatial_hp_cp = 0` (no extra high-pass before cellpose)
- `cellprob_threshold = -6`, `flow_threshold = 0`, `diameter = 4` (LBM defaults)

**Run this notebook with the `C:\Users\RBO\repos\mbo_utilities\.venv` kernel.**
Sibling notebook `run_clean_v2-7-7.ipynb` runs the same code in the v2-7-7 venv.

In [1]:
from pathlib import Path
from importlib.metadata import version
import numpy as np

import mbo_utilities as mbo
import lbm_suite2p_python as lsp

INPUT = Path(r"D:\demo\raw")
SAVE_PATH = Path(r"D:\demo\results_current_clean-lsp-shim")
PLANES = [1]

for pkg in ("mbo_utilities", "lbm_suite2p_python", "suite2p", "cellpose"):
    try:
        print(f"{pkg:<22} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:<22} <metadata missing>: {e}")

mbo_utilities          3.0.0
lbm_suite2p_python     3.0.1
suite2p                1.0.0.2.dev40+gc9af7c6e9
cellpose               4.1.1


In [2]:
# load lazy array; force phase corrections OFF on the array object
# (this bypasses GUI/CLI plumbing — the attribute is read by the reader
# at materialization time)
arr = mbo.imread(INPUT)
if hasattr(arr, "fix_phase"):
    arr.fix_phase = False
if hasattr(arr, "use_fft"):
    arr.use_fft = False
print(f"arr.fix_phase = {getattr(arr, 'fix_phase', '<no attr>')}")
print(f"arr.use_fft   = {getattr(arr, 'use_fft', '<no attr>')}")
print(f"arr.shape     = {getattr(arr, 'shape', None)}")

Counting frames:   0%|          | 0/2 [00:00<?, ?it/s]

arr.fix_phase = False
arr.use_fft   = False
arr.shape     = (1574, 1, 14, 550, 448)


In [3]:
# build ops with explicit overrides for everything we care about
ops = lsp.default_ops()
ops.update({
    "do_registration": False,
    "anatomical_only": 4,
    "tau": 1.3,
    "fs": 14.0,
    "spatial_hp_cp": 0.0,
    "diameter": 4,
    "cellprob_threshold": -6,
    "flow_threshold": 0,
    "sparse_mode": False,        # cellpose path, not sparsery
    "denoise": False,
    # disable npix_norm filter (matches mbo behavior)
    "npix_norm_min": -1.0,
    "npix_norm_max": float("inf"),
    # ensure phase-correction flags propagate via metadata too
    "fix_phase": False,
    "use_fft": False,
})
for k in ("do_registration", "anatomical_only", "tau", "fs", "spatial_hp_cp",
         "diameter", "cellprob_threshold", "flow_threshold", "sparse_mode",
         "npix_norm_min", "npix_norm_max", "fix_phase", "use_fft"):
    print(f"{k:<22} = {ops.get(k)}")

Importing suite2p packages...
do_registration        = False
anatomical_only        = 4
tau                    = 1.3
fs                     = 14.0
spatial_hp_cp          = 0.0
diameter               = 4
cellprob_threshold     = -6
flow_threshold         = 0
sparse_mode            = False
npix_norm_min          = -1.0
npix_norm_max          = inf
fix_phase              = False
use_fft                = False


In [4]:
ops_paths = lsp.pipeline(
    arr,
    save_path=SAVE_PATH,
    ops=ops,
    planes=PLANES,
    keep_raw=True,    # keep data_raw.bin so we can diff bytes later
    keep_reg=True,
)
print("\nwritten ops:")
for p in ops_paths:
    print(f"  {p}")

Delegating to run_volume (volumetric input detected)...
Processing 1 planes in volume (Total planes: 14)
Output: D:\demo\results_current_clean-lsp-shim

--- Volume Step: Plane 1 ---
Importing suite2p packages...
Writing binary to D:\demo\results_current_clean-lsp-shim\zplane01_tp00001-01574...
  Materializing data.bin from data_raw.bin
  Using full image dimensions for anatomical detection (avoids cropping issues)
  Computed meanImg from binary
  Computed meanImgE from meanImg


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 18 frames
suite2p.detection.detect: 100%|##########| 4/4 [00:00<00:00,  5.10it/s]
suite2p.detection.detect: Binned movie of size [85,550,448] created in 0.78 sec.
suite2p.detection.detect: >>>> CELLPOSE finding masks in max_proj
suite2p.detection.anatomical: !NOTE! diameter set to 4.00 for cell detection with cellpose
cellpose.core: ** TORCH CUDA version installed and working. **
cellpose.core: ** TORCH CUDA version installed and working. **
cellpose.core: >>>> using GPU (CUDA)
cellpose.models: >>>> loading model C:\Users\RBO\.cellpose\models\cpsam
suite2p.detection.anatomical: >>>> 242 masks detected, median diameter = 12.62 
suite2p.detection.anatomical: Detected 242 ROIs, 14.02 sec
suite2p.detection.detect: Detected

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 216 accepted / 26 rejected ROIs
regPC or regDX not found in ops, skipping PC metrics.

Genering volumetric statistics...

written ops:
  D:\demo\results_current_clean-lsp-shim\zplane01_tp00001-01574\ops.npy


In [5]:
# summarize results
for op_path in ops_paths:
    plane_dir = Path(op_path).parent
    op = np.load(op_path, allow_pickle=True).item()
    stat = np.load(plane_dir / "stat.npy", allow_pickle=True)
    iscell = np.load(plane_dir / "iscell.npy", allow_pickle=True)
    iscell_arr = iscell[:, 0] if iscell.ndim == 2 else iscell
    print(f"{plane_dir.name}")
    print(f"  detected: {len(stat)}")
    print(f"  accepted: {int(iscell_arr.sum())}")
    print(f"  rejected: {int((~iscell_arr.astype(bool)).sum())}")
    print(f"  fix_phase (saved in ops): {op.get('fix_phase')}")
    print(f"  use_fft   (saved in ops): {op.get('use_fft')}")
    print(f"  bin_size implied: round(tau*fs) = round({op.get('tau')}*{op.get('fs')}) = {round(op.get('tau', 0) * op.get('fs', 0))}")

zplane01_tp00001-01574
  detected: 242
  accepted: 216
  rejected: 26
  fix_phase (saved in ops): False
  use_fft   (saved in ops): False
  bin_size implied: round(tau*fs) = round(1.3*14.0) = 18


## Next step

Run `run_clean_v2-7-7.ipynb` (in the v2-7-7 venv) writing to `D:\demo\results_v2-7-7_clean`, then open `compare_runs.ipynb` with `OLD_DIR` and `NEW_DIR` pointing at the two `_clean` plane dirs. With `fix_phase=False` actually applied in both, the `data.bin` md5s should now match (or at minimum, the per-pixel diff should be 0).